# Linear Regression with HDB Geocoding

This notebook cleans raw rental records, joins HDB block attributes by a normalized address key, then uses the repository's postal-code geocoder from `src/transformers.py`. The geocoder derives postal sector, district, and region from `POSTAL_CODE` using `src/config/config_2026.yaml`.

## Join and geocoding design

A deterministic `TOWN | BLOCK | STREET` key is used because both files describe the same HDB block identity. The HDB reference is deduplicated and joined with `validate='many_to_one'`; this prevents accidental row multiplication. Coordinates and block attributes come from the HDB reference. Postal geocoding is applied after the join because postal code exists in the HDB file, not in the rental file. Invalid postal codes remain missing and are tracked through coverage diagnostics.

In [55]:
from pathlib import Path
import sys
import yaml
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'data' / 'train.csv').exists()), HERE)
GEO_ROOT = PROJECT_ROOT / 'models' / 'basic models' / 'linear regression + geo encoding'
GEO_SRC = GEO_ROOT / 'src'
sys.path.insert(0, str(GEO_ROOT))
from src.transformers import FloorTransformer, PostcodeRegionTransformer
with open(GEO_SRC / 'config' / 'config_2026.yaml', encoding='utf-8') as handle:
    geo_config = yaml.safe_load(handle)
postcode_geocoder = PostcodeRegionTransformer(geo_config)
floor_transformer = FloorTransformer(geo_config)
rent_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'train.csv')
hdb_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'sg-hdb-block.csv', dtype={'POSTAL_CODE': 'string'})
print('Rental data:', rent_raw.shape)
print('HDB reference:', hdb_raw.shape)

Rental data: (150000, 11)
HDB reference: (9464, 11)


## Cleaning

Text values are stripped, whitespace-collapsed, and lowercased. Common street-type spellings are canonicalized and flat-type variants are harmonized. `FURNISHED` is retained as a binary variable. The same function is applied to both datasets before constructing the join key.

In [56]:
STREET_REPLACEMENTS = {r'\bavenue\b':'ave', r'\broad\b':'rd', r'\bstreet\b':'st', r'\bdrive\b':'dr', r'\bcentral\b':'ctrl', r'\bplace\b':'pl', r'\bclose\b':'cl', r'\bcrescent\b':'cres', r'\blane\b':'ln', r'\bparkway\b':'pkwy', r'\bterrace\b':'ter', r'\bview\b':'vw'}
def normalize_text(s):
    return s.astype('string').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
def clean_common(frame):
    out = frame.copy()
    for col in out.select_dtypes(include=['object','string']).columns: out[col] = normalize_text(out[col])
    if 'STREET' in out:
        for pattern, replacement in STREET_REPLACEMENTS.items(): out['STREET'] = out['STREET'].str.replace(pattern, replacement, regex=True)
        out['STREET'] = out['STREET'].str.replace(r'\s+', ' ', regex=True).str.strip()
    if 'FLAT_TYPE' in out: out['FLAT_TYPE'] = out['FLAT_TYPE'].replace({'1 room':'1-room','2 room':'2-room','3 room':'3-room','4 room':'4-room','5 room':'5-room'})
    if 'FURNISHED' in out: out['FURNISHED'] = out['FURNISHED'].replace({'yes':1,'y':1,'true':1,'1':1,'no':0,'n':0,'false':0,'0':0}).astype('Int64')
    return out
rent = clean_common(rent_raw).drop_duplicates().reset_index(drop=True)
hdb = clean_common(hdb_raw)
rent['RENT_APPROVAL_DATE'] = pd.to_datetime(rent['RENT_APPROVAL_DATE'], format='%Y-%m', errors='coerce')
rent['RENT_YEAR'] = rent['RENT_APPROVAL_DATE'].dt.year
rent['RENT_MONTH'] = rent['RENT_APPROVAL_DATE'].dt.month
rent['RENT_QUARTER'] = rent['RENT_APPROVAL_DATE'].dt.quarter
rent['LEASE_AGE'] = rent['RENT_YEAR'] - pd.to_numeric(rent['LEASE_COMMENCE_DATE'], errors='coerce')
rent['ADDRESS_KEY'] = rent['TOWN'] + '|' + rent['BLOCK'] + '|' + rent['STREET']
hdb['ADDRESS_KEY'] = hdb['TOWN'] + '|' + hdb['BLOCK'] + '|' + hdb['STREET']

In [57]:
hdb = hdb.drop_duplicates('ADDRESS_KEY', keep='first')
geo_columns = ['ADDRESS_KEY','POSTAL_CODE','LATITUDE','LONGITUDE','MAX_FLOOR','YEAR_COMPLETED','SUBZONE','PLANNING_AREA','REGION']
rent_geo = rent.merge(hdb[geo_columns], on='ADDRESS_KEY', how='left', validate='many_to_one')
rent_geo['HDB_BLOCK_MATCHED'] = rent_geo['LATITUDE'].notna().astype('int8')
rent_geo['BUILDING_AGE'] = rent_geo['RENT_YEAR'] - pd.to_numeric(rent_geo['YEAR_COMPLETED'], errors='coerce')
rent_geo = postcode_geocoder.transform(rent_geo)
rent_geo = floor_transformer.transform(rent_geo)
print('HDB match rate:', f"{rent_geo['HDB_BLOCK_MATCHED'].mean():.2%}")
print('Postal geocoding coverage:', f"{rent_geo['POSTAL_DISTRICT'].notna().mean():.2%}")
rent_geo[['POSTAL_CODE','POSTAL_SECTOR','POSTAL_DISTRICT','POSTAL_REGION','LATITUDE','LONGITUDE']].head()

HDB match rate: 100.00%
Postal geocoding coverage: 96.50%


,POSTAL_CODE,POSTAL_SECTOR,POSTAL_DISTRICT,POSTAL_REGION,LATITUDE,LONGITUDE
0,821604,82,19,3,1.404310,103.907413
1,543290,54,19,3,1.397018,103.897535
2,270005,27,10,1,1.312693,103.788309
3,792436,79,28,3,1.392878,103.876976
4,752467,75,27,3,1.451141,103.813630


## Model and evaluation

The postal code itself is excluded to avoid treating it as a continuous numeric measurement. It is transformed into `POSTAL_SECTOR`, `POSTAL_DISTRICT`, and `POSTAL_REGION` by `PostcodeRegionTransformer`. These derived geography labels are nominal categories, so they are one-hot encoded: each category becomes a 0/1 indicator and the regression model does not assume that district 20 is twice district 10.

Latitude and longitude are retained as continuous numeric variables because distance and location gradients are meaningful. `MAX_FLOOR`, `FLOOR_NUM`, and `FLOOR_NUM_CAPPED` are also numeric because they represent ordered quantities. `FLOOR_BAND` is one-hot encoded because it represents named ranges rather than a precise linear effect. `IS_HIGH_FLOOR`, `IS_BASEMENT`, `FURNISHED`, and `HDB_BLOCK_MATCHED` are binary 0/1 indicators and remain numeric.

All numeric columns use median imputation, while categorical columns use the most frequent category followed by `OneHotEncoder(handle_unknown='ignore')`. These transformations are inside a scikit-learn `Pipeline` and `ColumnTransformer`, so they are fitted only on the training split and applied unchanged to the test split. This prevents information from the evaluation data influencing the encoding.

In [58]:
# Encoding choices: coordinates, floor numbers, building age, and binary flags
# remain numeric. Postal sector/district/region and FLOOR_BAND are nominal
# categories, so one-hot encoding avoids imposing a false linear ordering.
y = pd.to_numeric(rent_geo.pop('MONTHLY_RENT'), errors='coerce')
X = rent_geo.drop(columns=['RENT_APPROVAL_DATE','ADDRESS_KEY','FEE','POSTAL_CODE'], errors='ignore')

valid = y.notna(); X, y = X.loc[valid].copy(), y.loc[valid].copy()
X = X.replace(r'^\s*$', np.nan, regex=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
numeric_geo = ['FLOOR_AREA_SQM','YEAR_COMPLETED', 'BUILDING_AGE']
categorical_geo = ['POSTAL_SECTOR', 'POSTAL_DISTRICT', 'POSTAL_REGION', 'FLOOR_BAND', 'FLAT_TYPE', 'FLAT_MODEL', 'IS_HIGH_FLOOR', 'IS_BASEMENT', 'FURNISHED']
num = [col for col in numeric_geo if col in X_train.columns]
cat = [col for col in categorical_geo if col in X_train.columns]
for col in cat:
    X_train[col] = X_train[col].astype(object).where(X_train[col].notna(), np.nan)
    X_test[col] = X_test[col].astype(object).where(X_test[col].notna(), np.nan)
prep = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num),
                        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), 
                                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)], remainder='drop', verbose_feature_names_out=True)
model = Pipeline([('preprocessor', prep), ('regressor', LinearRegression())])
model.fit(X_train, y_train); predictions = model.predict(X_test)
metrics = pd.DataFrame({'metric':['MAE','RMSE','R²'], 'value':[mean_absolute_error(y_test,predictions), np.sqrt(mean_squared_error(y_test,predictions)), r2_score(y_test,predictions)]})
print(f'Train rows: {len(X_train):,}; test rows: {len(X_test):,}')
metrics

Train rows: 119,487; test rows: 29,872


,metric,value
0,MAE,405.945833
1,RMSE,528.507895
2,R²,0.503861


In [59]:
prep

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [60]:
evaluation = pd.DataFrame({
    'actual': y_test.to_numpy(),
    'predicted': predictions,
})
evaluation['residual'] = evaluation['actual'] - evaluation['predicted']
print('Prediction summary:')
print(evaluation.describe())
print('Mean residual:', evaluation['residual'].mean())
evaluation.head(10)

Prediction summary:
             actual     predicted      residual
count  29872.000000  29872.000000  29872.000000
mean    2725.215921   2716.529494      8.686427
std      750.338316    529.804317    528.445351
min      400.000000   1128.852404  -3674.897176
25%     2100.000000   2337.173627   -319.251795
50%     2700.000000   2706.452442    -10.035586
75%     3250.000000   3084.239272    340.040974
max     7600.000000   5252.162129   3786.630434
Mean residual: 8.686427012878132


,actual,predicted,residual
0,4900,3522.420433,1377.579567
1,3800,3064.922722,735.077278
2,2000,2826.639308,-826.639308
3,2400,2155.573063,244.426937
4,3700,3822.080218,-122.080218
5,3400,3313.010602,86.989398
6,2200,2594.795489,-394.795489
7,2200,2517.525144,-317.525144
8,3100,3931.468258,-831.468258
9,2350,2705.205135,-355.205135


## Why geospatial features can reduce linear-regression performance

Postal features can hurt holdout performance even when they are intuitively relevant. Postal districts are coarse and may duplicate information already represented by towns, planning areas, or coordinates. Latitude and longitude have nonlinear effects on rent, while ordinary linear regression assumes additive linear effects. One-hot categories can also create sparse, unstable coefficients, especially for detailed areas. Finally, geography may interact with flat type, floor area, lease age, and time, but ordinary linear regression does not learn these interactions automatically. These limitations motivate trying a nonlinear tree-based model while keeping the same feature selection and evaluation split.

## XGBoost continuation

XGBoost is given the same imputed and one-hot encoded matrix used by the linear model, ensuring the comparison is about the estimator rather than a different dataset. It can model nonlinear relationships and interactions between geographic, floor, and property features. The settings below are intentionally modest baseline values rather than a tuned final model.

In [61]:
from xgboost import XGBRegressor

X_train_encoded = model.named_steps['preprocessor'].transform(X_train)
X_test_encoded = model.named_steps['preprocessor'].transform(X_test)
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=2,
)
xgb_model.fit(X_train_encoded, y_train)
xgb_predictions = xgb_model.predict(X_test_encoded)
xgb_metrics = pd.DataFrame({
    'metric': ['MAE', 'RMSE', 'R²'],
    'linear_regression': [mean_absolute_error(y_test, predictions), np.sqrt(mean_squared_error(y_test, predictions)), r2_score(y_test, predictions)],
    'xgboost': [mean_absolute_error(y_test, xgb_predictions), np.sqrt(mean_squared_error(y_test, xgb_predictions)), r2_score(y_test, xgb_predictions)],
})
xgb_metrics

,metric,linear_regression,xgboost
0,MAE,405.945833,396.344971
1,RMSE,528.507895,509.885496
2,R²,0.503861,0.538209
